# 063 — Project: a Next-Word Predictor

Text generation is ordinary supervised learning in disguise: slide a window
through each sentence, everything before position $i$ is the input and the word
at $i$ is the label. Measured on real text, **56,526 words become 51,099
training pairs** — and with a 14,295-word vocabulary the embedding and softmax
are **96%** of the parameters while the LSTM is **4%**.

Downloads the 20 newsgroups corpus once (~14 MB).

| Part | What we check |
|---|---|
| A | sentences into input/output pairs |
| B | how much of the padded tensor is padding |
| C | where the parameters actually are |
| D | greedy generation, by hand |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — The n-gram expansion

In [ ]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups

docs = fetch_20newsgroups(subset="train",
                          remove=("headers", "footers", "quotes")).data[:400]
lines = [s.strip() for d in docs for s in d.split("\n") if len(s.split()) >= 3]

vocab = {}
for line in lines:
    for word in line.lower().split():
        vocab.setdefault(word, len(vocab) + 1)      # 0 is reserved for padding
V = len(vocab)

sequences = [[vocab[w] for w in line.lower().split()] for line in lines]
pairs = [(seq[:i], seq[i]) for seq in sequences for i in range(1, len(seq))]
total_words = sum(len(s) for s in sequences)

print(f"  {len(lines):,} lines, {total_words:,} words, vocabulary {V:,}")
print(f"  -> {len(pairs):,} training pairs ({len(pairs) / total_words:.2f} per word)")
assert (total_words, V, len(pairs)) == (56_526, 14_295, 51_099)

print("\n  the first sentence, expanded:")
first = sequences[0][:5]
inverse = {i: w for w, i in vocab.items()}
for i in range(1, len(first)):
    print(f"    {[inverse[t] for t in first[:i]]} -> {inverse[first[i]]}")
print("\n  Every pair shares a prefix with the next: the dataset is redundant by")
print("  construction, which is exactly what makes it big enough to train on.")

## Part B — Padding

In [ ]:
lengths = np.array([len(x) for x, _ in pairs])
maxlen = int(lengths.max())
print(f"  input length: mean {lengths.mean():.1f}, median {int(np.median(lengths))}, "
      f"max {maxlen}")
waste = 1 - lengths.sum() / (maxlen * len(pairs))
print(f"  padded to {maxlen}: {waste:.0%} of the tensor is padding")
assert maxlen == 42 and round(waste * 100) == 86

for cap in (10, 20, 42):
    used = np.minimum(lengths, cap).sum()
    kept = (lengths <= cap).mean()
    print(f"    cap at {cap:>3}: {1 - used / (cap * len(pairs)):>3.0%} padding, "
          f"{kept:>4.0%} of pairs untruncated")
print("\n  Truncating at 20 keeps almost every pair whole and halves the waste.")

## Part C — Where the weight sits

In [ ]:
EMBED, HIDDEN = 100, 150
embedding = V * EMBED
lstm = 4 * ((EMBED + HIDDEN) * HIDDEN + HIDDEN)
softmax = HIDDEN * V + V
total = embedding + lstm + softmax

for name, p in (("Embedding", embedding), ("LSTM(150)", lstm),
                ("Dense(softmax)", softmax)):
    print(f"  {name:<18}{p:>12,}  ({p / total:>5.1%})")
print(f"  {'TOTAL':<18}{total:>12,}")
vocab_facing = (embedding + softmax) / total
print(f"\n  vocabulary-facing: {vocab_facing:.1%}   recurrent: {lstm / total:.1%}")
assert round(vocab_facing * 100) == 96 and round(lstm / total * 100) == 4
print()
print("The same shape as the CNN dense head: a cheap extractor behind an")
print("expensive interface to a large discrete space.")

## Part D — Generation, by hand

Greedy decoding with a scripted "model" — a bigram count table built from the
same text. The point is the loop, not the quality.

In [ ]:
from collections import Counter, defaultdict

following = defaultdict(Counter)
for seq in sequences:
    for a, b in zip(seq, seq[1:]):
        following[a][b] += 1

def generate(seed_word, n=8):
    if seed_word not in vocab:
        return f"'{seed_word}' is not in the vocabulary"
    out = [seed_word]
    token = vocab[seed_word]
    for _ in range(n):
        if not following[token]:
            break
        token = following[token].most_common(1)[0][0]     # greedy: take the top
        out.append(inverse[token])
    return " ".join(out)

for seed in ("the", "i", "we"):
    print(f"  {seed!r:>6} -> {generate(seed)}")
print()
print("Greedy decoding loops: predict, append, feed back. It also gets stuck,")
print("which is why real generation samples instead of always taking the top")
print("token - and why lesson 067's sequence-to-sequence models need an <END>.")
assert generate("the").startswith("the ")

## What this notebook established

- 5,427 lines and **56,526 words** expanded to **51,099** input/output pairs,
  0.90 per word, by sliding a window through each sentence.
- Padding to the longest input (42) leaves **86%** of the tensor as zeros;
  truncating at 20 keeps almost every pair whole.
- With a 14,295-word vocabulary, the embedding and softmax are **96%** of the
  parameters and the LSTM is **4%**.
- Greedy decoding is a loop — predict, append, feed back — and it gets stuck,
  which is why sampling and an `<END>` token exist.

## Exercises

1. Cap the vocabulary at 5,000 words with an `<UNK>` token. How much of the
   model disappears, and what breaks?
2. Weight-tie the embedding and the softmax (same matrix). What is the new
   total, and what does the tying assume?
3. Replace greedy decoding with sampling from the top 5. Does it still get
   stuck?